# Relevant Python 3.13 Changes — Advanced Problems with Solutions

This notebook is an advanced, problem-driven companion to a Python-version-changes lesson. It focuses on **relevant Python 3.13 features**, production-minded usage, migration risks, and robust solution patterns.

**Target runtime:** CPython 3.13.x  
**Dependencies:** standard library only  
**Format:** every problem is immediately followed by a complete solution, tests, and best-practice notes.

> Python 3.13 was released on October 7, 2024. The free-threaded build and JIT compiler introduced in 3.13 are experimental and may not be present or enabled in a particular interpreter build.

## Learning goals

After completing the notebook, you should be able to:

1. Detect 3.13 runtime capabilities without assuming every CPython build is identical.
2. Write thread-safe code that is ready for free-threaded CPython.
3. Apply `copy.replace()`, strict batching, queue shutdown, and new asyncio APIs.
4. Explain the defined `locals()` semantics and the write-through `frame.f_locals` proxy.
5. Use Python 3.13 typing features: type-parameter defaults, `TypeIs`, `ReadOnly`, and protocol introspection.
6. Build safer deprecation paths with `warnings.deprecated()` and `argparse`.
7. Use Z85, optimized ASTs, class metadata, unnamed configuration sections, and the SQLite `dbm` backend.
8. Audit a codebase for removals from PEP 594.

## Official references

- [What’s New in Python 3.13](https://docs.python.org/3/whatsnew/3.13.html)
- [Python 3.13 library reference](https://docs.python.org/3.13/library/index.html)
- [PEP 703 — Making the GIL Optional](https://peps.python.org/pep-0703/)
- [PEP 744 — JIT Compilation](https://peps.python.org/pep-0744/)
- [PEP 667 — Consistent Views of Namespaces](https://peps.python.org/pep-0667/)
- [PEP 696 — Type Defaults](https://peps.python.org/pep-0696/)
- [PEP 702 — Marking Deprecations](https://peps.python.org/pep-0702/)
- [PEP 705 — Read-only TypedDict Items](https://peps.python.org/pep-0705/)
- [PEP 742 — TypeIs](https://peps.python.org/pep-0742/)
- [PEP 594 — Removing Dead Batteries](https://peps.python.org/pep-0594/)

## 0. Environment and reproducibility checks

In [1]:
import sys
import platform

if sys.version_info < (3, 13):
    raise RuntimeError(
        f"This notebook requires Python 3.13+, found {sys.version.split()[0]}"
    )

print("Python:", sys.version)
print("Implementation:", platform.python_implementation())
print("Platform:", platform.platform())

if sys.version_info[:2] != (3, 13):
    print("Note: this notebook targets 3.13; later versions may show evolved behavior.")

Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Implementation: CPython
Platform: Windows-10-10.0.19045-SP0


### Best-practice baseline

- Prefer **capability detection** (`hasattr`, documented configuration variables) over version-only checks.
- Treat experimental runtime features as optional.
- Keep examples deterministic: use assertions, bounded waits, explicit shutdown, and temporary directories.
- Do not infer performance from a single notebook timing.
- Remember that typing features generally guide static analyzers; they do not automatically enforce runtime immutability.

## Problem 1 — Build a Python 3.13 capability report

Create a function that reports:

- whether the interpreter build supports free-threading;
- whether the GIL is currently enabled;
- whether a JIT introspection API exists and whether the JIT is enabled;
- the process-usable CPU count and host CPU count;
- whether the new REPL was explicitly disabled.

The solution must work even when optional CPython-only APIs are absent.

In [2]:
from __future__ import annotations

import os
import sys
import sysconfig
from pprint import pprint
from typing import Any


def python_313_capabilities() -> dict[str, Any]:
    """Return a defensive capability report for the current interpreter."""
    gil_probe = getattr(sys, "_is_gil_enabled", None)
    jit_api = getattr(sys, "_jit", None)

    report: dict[str, Any] = {
        "implementation": sys.implementation.name,
        "version": sys.version.split()[0],
        "free_threaded_build": sysconfig.get_config_var("Py_GIL_DISABLED") == 1,
        "gil_enabled": gil_probe() if callable(gil_probe) else None,
        "jit_api_available": jit_api is not None,
        "jit_enabled": (
            jit_api.is_enabled()
            if jit_api is not None and hasattr(jit_api, "is_enabled")
            else None
        ),
        "process_cpu_count": (
            os.process_cpu_count() if hasattr(os, "process_cpu_count") else None
        ),
        "host_cpu_count": os.cpu_count(),
        "basic_repl_requested": bool(os.environ.get("PYTHON_BASIC_REPL")),
    }
    return report


capabilities = python_313_capabilities()
pprint(capabilities, sort_dicts=False)

assert capabilities["implementation"]
assert capabilities["version"].startswith(("3.13", "3.14", "3.15", "3.16"))

{'implementation': 'cpython',
 'version': '3.13.7',
 'free_threaded_build': False,
 'gil_enabled': True,
 'jit_api_available': False,
 'jit_enabled': None,
 'process_cpu_count': 8,
 'host_cpu_count': 8,
 'basic_repl_requested': False}


**Why this is robust:** `sys._is_gil_enabled()` and `sys._jit` are implementation details that are not guaranteed on every Python implementation or build. `sysconfig.get_config_var("Py_GIL_DISABLED")` detects build support, while the GIL probe reports current runtime state.

## Problem 2 — Design a free-threading-ready CPU task

Implement parallel sum-of-squares without shared mutable state. Then implement a shared counter correctly using a lock.

The code must be correct on both conventional GIL builds and experimental free-threaded builds. Do **not** assume that operations such as `counter += 1` are a sufficient synchronization strategy.

In [3]:
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from threading import Lock
from collections.abc import Iterable, Sequence


def chunked(values: Sequence[int], parts: int) -> list[Sequence[int]]:
    if parts < 1:
        raise ValueError("parts must be positive")
    size = max(1, (len(values) + parts - 1) // parts)
    return [values[i : i + size] for i in range(0, len(values), size)]


def local_sum_of_squares(values: Iterable[int]) -> int:
    return sum(value * value for value in values)


def parallel_sum_of_squares(values: Sequence[int], workers: int = 4) -> int:
    """Parallelize independent chunks; combine results in the caller thread."""
    pieces = chunked(values, workers)
    with ThreadPoolExecutor(max_workers=workers) as executor:
        return sum(executor.map(local_sum_of_squares, pieces))


values = list(range(25_000))
serial = local_sum_of_squares(values)
parallel = parallel_sum_of_squares(values, workers=4)
assert parallel == serial
print("sum-of-squares:", parallel)

sum-of-squares: 5208020837500


In [4]:
@dataclass
class SafeCounter:
    value: int = 0
    _lock: Lock = field(default_factory=Lock, init=False, repr=False)

    def increment(self, amount: int = 1) -> None:
        if amount < 0:
            raise ValueError("amount must be non-negative")
        with self._lock:
            self.value += amount


def increment_many(counter: SafeCounter, repetitions: int) -> None:
    for _ in range(repetitions):
        counter.increment()


counter = SafeCounter()
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(increment_many, counter, 2_000) for _ in range(8)]
    for future in futures:
        future.result()

assert counter.value == 16_000
print("synchronized counter:", counter.value)

synchronized counter: 16000


**Best practice:** prefer ownership and result aggregation over fine-grained shared mutation. When state really is shared, protect the complete invariant with a synchronization primitive.

## Problem 3 — Benchmark without making false JIT claims

Create a small timing harness that:

- warms up the workload;
- uses repeated measurements;
- reports the median;
- records JIT capability separately from timing;
- avoids querying JIT active-state APIs inside the hot loop.

The result is an observation for this machine and build, not a universal performance conclusion.

In [5]:
import statistics
import sys
import timeit


def polynomial_workload(n: int = 20_000) -> int:
    total = 0
    for x in range(n):
        total += (x * x + 3 * x + 7) % 97
    return total


def jit_status() -> dict[str, object]:
    jit = getattr(sys, "_jit", None)
    return {
        "api_available": jit is not None,
        "enabled": jit.is_enabled() if jit is not None else None,
    }


# Warm-up: useful for caches, specialization, and JIT-enabled builds.
for _ in range(20):
    polynomial_workload(5_000)

samples = timeit.repeat(
    stmt="polynomial_workload(10_000)",
    globals=globals(),
    repeat=7,
    number=20,
)

print("JIT status:", jit_status())
print("samples (seconds):", [round(value, 6) for value in samples])
print("median (seconds):", round(statistics.median(samples), 6))
assert polynomial_workload(100) == polynomial_workload(100)

JIT status: {'api_available': False, 'enabled': None}
samples (seconds): [0.031873, 0.033971, 0.0338, 0.041045, 0.03132, 0.028788, 0.027539]
median (seconds): 0.031873


## Problem 4 — Use `copy.replace()` as a uniform immutable-update API

Update a frozen dataclass, a named tuple, a `datetime`, and a user-defined immutable class through one API. The custom class must reject unknown fields and preserve validation.

In [6]:
from collections import namedtuple
from copy import replace
from dataclasses import dataclass
from datetime import datetime, timezone


@dataclass(frozen=True, slots=True)
class JobConfig:
    retries: int
    timeout: float
    region: str


Point = namedtuple("Point", "x y")


class Endpoint:
    __slots__ = ("host", "port", "tls")

    def __init__(self, host: str, port: int, tls: bool = True) -> None:
        if not host:
            raise ValueError("host cannot be empty")
        if not 1 <= port <= 65_535:
            raise ValueError("port must be in 1..65535")
        object.__setattr__(self, "host", host)
        object.__setattr__(self, "port", port)
        object.__setattr__(self, "tls", tls)

    def __replace__(self, /, **changes: object) -> "Endpoint":
        allowed = {"host", "port", "tls"}
        unknown = changes.keys() - allowed
        if unknown:
            raise TypeError(f"unknown fields: {sorted(unknown)}")
        values = {
            "host": self.host,
            "port": self.port,
            "tls": self.tls,
        }
        values.update(changes)
        return type(self)(**values)

    def __repr__(self) -> str:
        return f"Endpoint(host={self.host!r}, port={self.port}, tls={self.tls})"


config = JobConfig(retries=3, timeout=2.5, region="eu-central")
config_v2 = replace(config, timeout=5.0)

point = Point(10, 20)
point_v2 = replace(point, y=99)

stamp = datetime(2026, 1, 1, 12, tzinfo=timezone.utc)
stamp_v2 = replace(stamp, hour=18)

endpoint = Endpoint("api.example.com", 443)
endpoint_v2 = replace(endpoint, port=8443)

print(config_v2)
print(point_v2)
print(stamp_v2)
print(endpoint_v2)

assert config.timeout == 2.5 and config_v2.timeout == 5.0
assert point_v2 == Point(10, 99)
assert stamp_v2.hour == 18
assert endpoint.port == 443 and endpoint_v2.port == 8443

JobConfig(retries=3, timeout=5.0, region='eu-central')
Point(x=10, y=99)
2026-01-01 18:00:00+00:00
Endpoint(host='api.example.com', port=8443, tls=True)


In [7]:
try:
    replace(endpoint, protocol="https")
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)
else:
    raise AssertionError("unknown fields must be rejected")

TypeError: unknown fields: ['protocol']


## Problem 5 — Validate fixed-size records with `itertools.batched(strict=True)`

Decode a stream of integers into four-field telemetry records. The input may be a one-shot iterator, so the solution must not pre-consume it merely to check its length.

In [8]:
from dataclasses import dataclass
from itertools import batched
from collections.abc import Iterable


@dataclass(frozen=True, slots=True)
class TelemetryRecord:
    sensor_id: int
    timestamp: int
    value: int
    quality: int


def decode_telemetry(numbers: Iterable[int]) -> list[TelemetryRecord]:
    records: list[TelemetryRecord] = []
    for sensor_id, timestamp, value, quality in batched(
        numbers, 4, strict=True
    ):
        if quality not in range(0, 101):
            raise ValueError(f"invalid quality: {quality}")
        records.append(TelemetryRecord(sensor_id, timestamp, value, quality))
    return records


stream = (number for number in [7, 1000, 42, 98, 8, 1001, 43, 97])
records = decode_telemetry(stream)
print(records)
assert len(records) == 2

[TelemetryRecord(sensor_id=7, timestamp=1000, value=42, quality=98), TelemetryRecord(sensor_id=8, timestamp=1001, value=43, quality=97)]


In [9]:
for bad_input in ([7, 1000, 42], [7, 1000, 42, 120]):
    try:
        decode_telemetry(iter(bad_input))
    except ValueError as exc:
        print(f"{bad_input!r} -> {type(exc).__name__}: {exc}")

[7, 1000, 42] -> ValueError: batched(): incomplete batch
[7, 1000, 42, 120] -> ValueError: invalid quality: 120


**Best practice:** use strict mode when incomplete final batches represent corrupt or truncated input. Use non-strict batching only when a shorter final batch is explicitly valid.

## Problem 6 — Reason correctly about `locals()` snapshots (PEP 667)

Inside a function, demonstrate that:

1. mutating the mapping returned by `locals()` does not change fast local variables;
2. repeated calls return independent snapshots;
3. `exec()` without explicit namespaces does not provide a reliable channel for creating later-visible locals;
4. an explicit namespace reliably captures executed results.

In [10]:
def locals_snapshot_demo() -> dict[str, object]:
    x = 10

    first_snapshot = locals()
    first_snapshot["x"] = 999
    x_after_snapshot_mutation = x

    exec("generated = x + 5")
    second_snapshot = locals()

    explicit_namespace = {"x": x}
    exec("generated = x + 5", globals={}, locals=explicit_namespace)

    return {
        "x_after_snapshot_mutation": x_after_snapshot_mutation,
        "first_snapshot_x": first_snapshot["x"],
        "second_snapshot_x": second_snapshot["x"],
        "generated_visible_in_second_snapshot": "generated" in second_snapshot,
        "explicit_generated": explicit_namespace["generated"],
        "snapshots_are_same_object": first_snapshot is second_snapshot,
    }


result = locals_snapshot_demo()
print(result)

assert result["x_after_snapshot_mutation"] == 10
assert result["first_snapshot_x"] == 999
assert result["second_snapshot_x"] == 10
assert result["generated_visible_in_second_snapshot"] is False
assert result["explicit_generated"] == 15
assert result["snapshots_are_same_object"] is False

{'x_after_snapshot_mutation': 10, 'first_snapshot_x': 999, 'second_snapshot_x': 10, 'generated_visible_in_second_snapshot': False, 'explicit_generated': 15, 'snapshots_are_same_object': False}


## Problem 7 — Update a live frame through `frame.f_locals`

Build a minimal tracing tool that changes a local variable in a running function. Restore the previous trace function even if execution fails.

This demonstrates the Python 3.13 write-through proxy semantics intended for debuggers and similar tooling. Application code should not normally rewrite another frame’s locals.

In [11]:
import sys
from types import FrameType
from typing import Any


def target_function() -> int:
    score = 1
    marker = "safe point"
    return score


def score_rewriter(frame: FrameType, event: str, arg: Any):
    if (
        frame.f_code is target_function.__code__
        and event == "line"
        and frame.f_locals.get("score") == 1
    ):
        frame.f_locals["score"] = 42
    return score_rewriter


previous_trace = sys.gettrace()
sys.settrace(score_rewriter)
try:
    rewritten_score = target_function()
finally:
    sys.settrace(previous_trace)

print("rewritten score:", rewritten_score)
assert rewritten_score == 42

rewritten score: 42


## Problem 8 — Add defaults to type parameters (PEP 696)

Create a generic cache whose value type defaults to `str`. Introspect which parameter has no default and verify that subscripting with one type argument fills in the default.

In [12]:
from typing import NoDefault


class Cache[K, V = str]:
    def __init__(self) -> None:
        self._data: dict[K, V] = {}

    def put(self, key: K, value: V) -> None:
        self._data[key] = value

    def get(self, key: K) -> V:
        return self._data[key]


StringCache = Cache[int]
FloatCache = Cache[int, float]

print("StringCache:", StringCache)
print("FloatCache:", FloatCache)
print("StringCache args:", StringCache.__args__)

parameters = Cache.__type_params__
for parameter in parameters:
    print(parameter, "default =", parameter.__default__)

assert parameters[0].__default__ is NoDefault
assert parameters[1].__default__ is str
assert StringCache.__args__ == (int, str)

StringCache: __main__.Cache[int, str]
FloatCache: __main__.Cache[int, float]
StringCache args: (<class 'int'>, <class 'str'>)
K default = typing.NoDefault
V default = <class 'str'>


In [13]:
string_cache = Cache[int]()
string_cache.put(1, "ready")
assert string_cache.get(1) == "ready"

float_cache = Cache[int, float]()
float_cache.put(1, 3.5)
assert float_cache.get(1) == 3.5

print(string_cache._data, float_cache._data)

{1: 'ready'} {1: 3.5}


## Problem 9 — Combine `TypeIs` and `ReadOnly` for safer boundaries

Define an event payload where the identifier is read-only to static type checkers. Write a user-defined type predicate that narrows `object` to that payload type.

Then show the important distinction: `ReadOnly` is a static contract and does not freeze a runtime dictionary.

In [14]:
from typing import ReadOnly, TypeIs, TypedDict, get_type_hints


class EventPayload(TypedDict):
    event_id: ReadOnly[str]
    attempts: int
    payload: bytes


def is_event_payload(value: object) -> TypeIs[EventPayload]:
    if not isinstance(value, dict):
        return False
    return (
        isinstance(value.get("event_id"), str)
        and isinstance(value.get("attempts"), int)
        and isinstance(value.get("payload"), bytes)
    )


def process_unknown(value: object) -> str:
    if is_event_payload(value):
        # Static type checkers narrow value to EventPayload in this branch.
        return f"{value['event_id']}:{len(value['payload'])}"
    return "invalid"


candidate: object = {
    "event_id": "evt-001",
    "attempts": 2,
    "payload": b"abc",
}

print(process_unknown(candidate))
print(get_type_hints(EventPayload, include_extras=True))
assert process_unknown(candidate) == "evt-001:3"

evt-001:3
{'event_id': typing.ReadOnly[str], 'attempts': <class 'int'>, 'payload': <class 'bytes'>}


In [15]:
# Runtime dictionaries remain mutable; ReadOnly is enforced by type checkers.
runtime_payload: EventPayload = {
    "event_id": "evt-002",
    "attempts": 0,
    "payload": b"data",
}
runtime_payload["event_id"] = "mutated-at-runtime"  # type checker should reject
print(runtime_payload)

# Production boundary: expose a read-only Mapping when runtime mutation matters.
from types import MappingProxyType

read_only_view = MappingProxyType(runtime_payload)
try:
    read_only_view["event_id"] = "blocked"
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

{'event_id': 'mutated-at-runtime', 'attempts': 0, 'payload': b'data'}
TypeError: 'mappingproxy' object does not support item assignment


## Problem 10 — Introspect protocols in a plugin registry

Use `typing.is_protocol()` and `typing.get_protocol_members()` to reject non-protocol declarations and report the required member names for a plugin contract.

In [16]:
from typing import Protocol, get_protocol_members, is_protocol, runtime_checkable


@runtime_checkable
class Serializer(Protocol):
    media_type: str

    def dumps(self, value: object) -> bytes: ...
    def loads(self, payload: bytes) -> object: ...


class JsonLikeSerializer:
    media_type = "application/x-demo-json"

    def dumps(self, value: object) -> bytes:
        return repr(value).encode()

    def loads(self, payload: bytes) -> object:
        return payload.decode()


def describe_protocol(candidate: type[object]) -> frozenset[str]:
    if not is_protocol(candidate):
        raise TypeError(f"{candidate.__name__} is not a Protocol")
    return get_protocol_members(candidate)


members = describe_protocol(Serializer)
print("required members:", sorted(members))
print("runtime compatible:", isinstance(JsonLikeSerializer(), Serializer))

assert members == frozenset({"media_type", "dumps", "loads"})
assert isinstance(JsonLikeSerializer(), Serializer)

required members: ['dumps', 'loads', 'media_type']
runtime compatible: True


In [17]:
try:
    describe_protocol(JsonLikeSerializer)
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

TypeError: JsonLikeSerializer is not a Protocol


## Problem 11 — Create a type-checker-visible and runtime-visible deprecation

Decorate a legacy function, capture its runtime warning, inspect its `__deprecated__` metadata, and provide a non-deprecated replacement.

In [18]:
import warnings


@warnings.deprecated(
    "legacy_total() is deprecated; use total() instead",
    category=DeprecationWarning,
    stacklevel=1,
)
def legacy_total(values: list[int]) -> int:
    return sum(values)


def total(values: list[int]) -> int:
    return sum(values)


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    answer = legacy_total([1, 2, 3])

print("answer:", answer)
print("metadata:", legacy_total.__deprecated__)
print("warning:", caught[0].message)

assert answer == 6
assert len(caught) == 1
assert issubclass(caught[0].category, DeprecationWarning)
assert "use total" in legacy_total.__deprecated__

answer: 6
metadata: legacy_total() is deprecated; use total() instead


**Best practice:** use deprecation tests with warnings enabled. `DeprecationWarning` is commonly filtered outside `__main__`, so CI should run with an appropriate warning policy such as `-W default` or `-W error` for selected modules.

## Problem 12 — Deprecate a command-line option with `argparse`

Build a parser that accepts both `--format` and a deprecated `--legacy-format`. Capture the warning sent to standard error, and normalize both options to one internal field.

In [19]:
import argparse
import io
from contextlib import redirect_stderr


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(prog="report")
    parser.add_argument("--format", choices=("json", "text"), default="json")
    parser.add_argument(
        "--legacy-format",
        dest="format",
        choices=("json", "text"),
        deprecated=True,
    )
    return parser


parser = build_parser()
stderr = io.StringIO()
with redirect_stderr(stderr):
    namespace = parser.parse_args(["--legacy-format", "text"])

print("parsed:", namespace)
print("stderr:", stderr.getvalue().strip())

assert namespace.format == "text"
assert "deprecated" in stderr.getvalue().lower()

parsed: Namespace(format='text')
stderr: report: warning: option '--legacy-format' is deprecated


## Problem 13 — Correlate results with original tasks using async `as_completed()`

Run several jobs with different delays. Iterate asynchronously over `asyncio.as_completed()` and use each original task’s name to preserve correlation metadata.

In [20]:
import asyncio


async def delayed_job(label: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return label.upper()


async def completion_demo() -> list[tuple[str, str]]:
    tasks = [
        asyncio.create_task(delayed_job("alpha", 0.030), name="job-alpha"),
        asyncio.create_task(delayed_job("beta", 0.010), name="job-beta"),
        asyncio.create_task(delayed_job("gamma", 0.020), name="job-gamma"),
    ]

    completed: list[tuple[str, str]] = []
    async for original_task in asyncio.as_completed(tasks):
        assert original_task in tasks
        completed.append((original_task.get_name(), await original_task))
    return completed


completion_order = await completion_demo()
print(completion_order)
assert [name for name, _ in completion_order] == [
    "job-beta",
    "job-gamma",
    "job-alpha",
]

[('job-beta', 'BETA'), ('job-gamma', 'GAMMA'), ('job-alpha', 'ALPHA')]


## Problem 14 — Gracefully terminate an `asyncio.Queue`

Implement a producer/consumer pipeline without sentinel values. The producer must call `shutdown()`, the consumer must drain queued work, call `task_done()` exactly once per item, and stop on `QueueShutDown`.

In [21]:
async def async_queue_pipeline() -> list[int]:
    work: asyncio.Queue[int] = asyncio.Queue(maxsize=3)
    processed: list[int] = []

    async def producer() -> None:
        for item in range(8):
            await work.put(item)
        work.shutdown()  # graceful: existing items remain available

    async def consumer() -> None:
        while True:
            try:
                item = await work.get()
            except asyncio.QueueShutDown:
                break
            try:
                await asyncio.sleep(0)
                processed.append(item * item)
            finally:
                work.task_done()

    async with asyncio.TaskGroup() as group:
        group.create_task(producer())
        group.create_task(consumer())

    await work.join()
    return processed


async_results = await async_queue_pipeline()
print(async_results)
assert async_results == [value * value for value in range(8)]

[0, 1, 4, 9, 16, 25, 36, 49]


**Best practice:** reserve `shutdown(immediate=True)` for abort semantics. Immediate shutdown can violate the usual `join()` invariant because unfinished work may be unblocked without being processed.

## Problem 15 — Gracefully terminate a threaded `queue.Queue`

Create a two-worker threaded pipeline using `queue.Queue.shutdown()` and `queue.ShutDown`. Use bounded joins and ensure each dequeued item calls `task_done()`.

In [22]:
import queue
import threading


def threaded_queue_pipeline(items: list[int]) -> list[int]:
    work: queue.Queue[int] = queue.Queue()
    results: list[int] = []
    results_lock = threading.Lock()

    for item in items:
        work.put(item)

    def worker() -> None:
        while True:
            try:
                item = work.get()
            except queue.ShutDown:
                return
            try:
                result = item * 10
                with results_lock:
                    results.append(result)
            finally:
                work.task_done()

    threads = [
        threading.Thread(target=worker, name=f"worker-{index}")
        for index in range(2)
    ]
    for thread in threads:
        thread.start()

    work.shutdown()
    work.join()

    for thread in threads:
        thread.join(timeout=2)
        if thread.is_alive():
            raise RuntimeError(f"{thread.name} did not terminate")

    return sorted(results)


threaded_results = threaded_queue_pipeline(list(range(10)))
print(threaded_results)
assert threaded_results == [value * 10 for value in range(10)]

[0, 10, 20, 30, 40, 50, 60, 70, 80, 90]


## Problem 16 — Parse multiple line endings with `StreamReader.readuntil()`

Read a byte stream containing both CRLF and LF records by passing a tuple of separators. Strip the matched terminator without corrupting the record body.

In [23]:
async def parse_mixed_line_endings(payload: bytes) -> list[bytes]:
    reader = asyncio.StreamReader()
    reader.feed_data(payload)
    reader.feed_eof()

    separators = (b"\r\n", b"\n")
    records: list[bytes] = []

    while not reader.at_eof():
        try:
            framed = await reader.readuntil(separators)
        except asyncio.IncompleteReadError as exc:
            if exc.partial:
                records.append(exc.partial)
            break

        if framed.endswith(b"\r\n"):
            records.append(framed[:-2])
        else:
            records.append(framed[:-1])

    return records


mixed_records = await parse_mixed_line_endings(b"alpha\r\nbeta\ngamma")
print(mixed_records)
assert mixed_records == [b"alpha", b"beta", b"gamma"]

[b'alpha', b'beta', b'gamma']


## Problem 17 — Build a length-safe Z85 envelope

`base64.z85encode()` requires input lengths divisible by four. Design an envelope that stores the original payload length, pads safely, round-trips arbitrary bytes, and rejects impossible declared lengths.

In [24]:
import base64


def z85_pack(payload: bytes) -> bytes:
    if len(payload) >= 2**32:
        raise ValueError("payload is too large for a 32-bit length prefix")
    framed = len(payload).to_bytes(4, "big") + payload
    padding = (-len(framed)) % 4
    framed += b"\x00" * padding
    return base64.z85encode(framed)


def z85_unpack(encoded: bytes) -> bytes:
    framed = base64.z85decode(encoded)
    if len(framed) < 4:
        raise ValueError("missing length prefix")
    size = int.from_bytes(framed[:4], "big")
    available = len(framed) - 4
    if size > available:
        raise ValueError(
            f"declared payload size {size} exceeds available bytes {available}"
        )
    return framed[4 : 4 + size]


examples = [b"", b"a", b"abc", b"arbitrary-length payload", bytes(range(17))]
for payload in examples:
    encoded = z85_pack(payload)
    decoded = z85_unpack(encoded)
    print(len(payload), encoded, decoded)
    assert decoded == payload

0 b'00000' b''
1 b'00001ve{oc' b'a'
3 b'00003vpAZD' b'abc'
24 b'0000ovrb*>BAg/gC}A&zzFa68aAg+my&r-)' b'arbitrary-length payload'
17 b'0000h009c61o!#m2NH?C3>iWS5c8Xg' b'\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\x0c\r\x0e\x0f\x10'


## Problem 18 — Analyze an optimized AST

Use Python 3.13’s optimized AST support to compare raw and optimized syntax trees. Build a metric that counts arithmetic `BinOp` nodes and verify that constant folding changes the optimized tree.

In [25]:
import ast


SOURCE = (
    "def calculate():\n"
    "    'Example function.'\n"
    "    assert 2 + 2 == 4\n"
    "    return (10 * 20) + (3 - 1)\n"
)


def count_nodes(tree: ast.AST, node_type: type[ast.AST]) -> int:
    return sum(isinstance(node, node_type) for node in ast.walk(tree))


raw_tree = ast.parse(SOURCE, mode="exec", optimize=0)
optimized_tree = ast.parse(SOURCE, mode="exec", optimize=1)

raw_binops = count_nodes(raw_tree, ast.BinOp)
optimized_binops = count_nodes(optimized_tree, ast.BinOp)

print("raw BinOp count:", raw_binops)
print("optimized BinOp count:", optimized_binops)
print(ast.dump(optimized_tree, indent=2))

assert raw_binops > optimized_binops

raw BinOp count: 4
optimized BinOp count: 0
Module(
  body=[
    FunctionDef(
      name='calculate',
      args=arguments(),
      body=[
        Expr(
          value=Constant(value='Example function.')),
        Assert(
          test=Compare(
            left=Constant(value=4),
            ops=[
              Eq()],
            comparators=[
              Constant(value=4)])),
        Return(
          value=Constant(value=202))])])


**Tooling note:** choose raw or optimized ASTs deliberately. Source linters often need the unoptimized tree, while compiler-oriented analyses may prefer the optimized representation.

## Problem 19 — Audit class assignments with new class metadata

Use `__static_attributes__`, `__firstlineno__`, and `property.__name__` to build a lightweight audit for slotted classes. Detect assignments to `self.<name>` that are missing from `__slots__` before those methods are executed.

In [26]:
class Account:
    __slots__ = ("balance", "audit")

    def __init__(self, balance: int) -> None:
        self.balance = balance
        self.audit = []

    @property
    def available(self) -> int:
        return self.balance

    def deposit(self, amount: int) -> None:
        self.balance += amount
        self.audit.append(("deposit", amount))

    def buggy_cache(self) -> None:
        self.cached_total = self.balance  # missing from __slots__


def slot_assignment_gaps(cls: type[object]) -> set[str]:
    slots = cls.__slots__
    if isinstance(slots, str):
        declared = {slots}
    else:
        declared = set(slots)
    assigned = set(getattr(cls, "__static_attributes__", ()))
    return assigned - declared


print("static assignments:", Account.__static_attributes__)
print("first line:", Account.__firstlineno__)
print("property name:", Account.available.__name__)
print("slot gaps:", slot_assignment_gaps(Account))

assert slot_assignment_gaps(Account) == {"cached_total"}
assert Account.available.__name__ == "available"

static assignments: ('audit', 'balance', 'cached_total')
first line: 1
property name: available
slot gaps: {'cached_total'}


## Problem 20 — Parse configuration with an unnamed top-level section

Parse a configuration format that has global key/value pairs before named sections. Validate required global fields and convert their values to application types.

In [27]:
import configparser


CONFIG_TEXT = (
    "host = 127.0.0.1\n"
    "port = 8080\n"
    "debug = no\n"
    "\n"
    "[database]\n"
    "name = analytics\n"
    "pool_size = 5\n"
)


def load_config(text: str) -> dict[str, object]:
    parser = configparser.ConfigParser(allow_unnamed_section=True)
    parser.read_string(text)

    global_section = parser[configparser.UNNAMED_SECTION]
    required = {"host", "port", "debug"}
    missing = required - global_section.keys()
    if missing:
        raise ValueError(f"missing global settings: {sorted(missing)}")

    return {
        "host": global_section["host"],
        "port": global_section.getint("port"),
        "debug": global_section.getboolean("debug"),
        "database": dict(parser["database"]),
    }


loaded_config = load_config(CONFIG_TEXT)
print(loaded_config)
assert loaded_config["port"] == 8080
assert loaded_config["debug"] is False

{'host': '127.0.0.1', 'port': 8080, 'debug': False, 'database': {'name': 'analytics', 'pool_size': '5'}}


## Problem 21 — Choose worker counts with `os.process_cpu_count()`

Create a conservative worker-budget function that respects CPU availability assigned to the current process. Handle `None`, invalid overrides, and I/O-bound expansion with an explicit cap.

In [28]:
import os


def worker_budget(
    *,
    cpu_bound: bool,
    override: int | None = None,
    io_multiplier: int = 4,
    hard_cap: int = 32,
) -> int:
    if override is not None:
        if override < 1:
            raise ValueError("override must be positive")
        return min(override, hard_cap)

    usable = os.process_cpu_count() or 1
    if cpu_bound:
        return min(usable, hard_cap)
    return min(usable * io_multiplier, hard_cap)


print("CPU-bound budget:", worker_budget(cpu_bound=True))
print("I/O-bound budget:", worker_budget(cpu_bound=False))
print("override budget:", worker_budget(cpu_bound=True, override=100))

assert worker_budget(cpu_bound=True) >= 1
assert worker_budget(cpu_bound=True, override=100) == 32

CPU-bound budget: 8
I/O-bound budget: 32
override budget: 32


## Problem 22 — Verify the new SQLite-backed `dbm` default

Create a temporary key/value database through the generic `dbm.open()` API, round-trip binary values, and inspect which backend was selected. Keep the example isolated and automatically cleaned up.

In [29]:
import dbm
import tempfile
from pathlib import Path


with tempfile.TemporaryDirectory() as temporary_directory:
    database_path = str(Path(temporary_directory) / "cache")

    with dbm.open(database_path, "c") as database:
        database[b"language"] = b"Python"
        database[b"version"] = b"3.13"

    selected_backend = dbm.whichdb(database_path)

    with dbm.open(database_path, "r") as database:
        restored = {
            key.decode(): database[key].decode()
            for key in database.keys()
        }

print("backend:", selected_backend)
print("restored:", restored)
assert restored == {"language": "Python", "version": "3.13"}
assert selected_backend is not None

backend: dbm.sqlite3
restored: {'language': 'Python', 'version': '3.13'}


On standard Python 3.13 installations with SQLite support, the generic default is normally `dbm.sqlite3`. The assertion intentionally checks functionality rather than hard-coding a platform packaging assumption.

## Problem 23 — Audit removals from PEP 594

Build a migration report for the 19 standard-library modules removed in Python 3.13. Verify they are absent from `sys.stdlib_module_names` and provide replacement directions for common cases.

In [30]:
import sys
from pprint import pprint


REMOVED_313_MODULES = {
    "aifc", "audioop", "cgi", "cgitb", "chunk", "crypt", "imghdr",
    "mailcap", "msilib", "nis", "nntplib", "ossaudiodev", "pipes",
    "sndhdr", "spwd", "sunau", "telnetlib", "uu", "xdrlib",
}

REPLACEMENT_GUIDANCE = {
    "cgi": "Use urllib.parse for query parsing and email.message for MIME data.",
    "imghdr": "Use a maintained file-type library or parse trusted formats explicitly.",
    "pipes": "Use shlex.quote and subprocess with argument lists.",
    "telnetlib": "Use a maintained Telnet client package or replace the protocol.",
    "nntplib": "Use a maintained third-party NNTP client if the protocol is required.",
    "audioop": "Use maintained audio-processing packages or format-specific tooling.",
}


def pep594_audit(import_names: set[str]) -> dict[str, object]:
    requested_removed = sorted(import_names & REMOVED_313_MODULES)
    unexpectedly_in_stdlib = sorted(
        REMOVED_313_MODULES & set(sys.stdlib_module_names)
    )
    return {
        "removed_imports_found": requested_removed,
        "unexpectedly_still_in_stdlib": unexpectedly_in_stdlib,
        "guidance": {
            name: REPLACEMENT_GUIDANCE.get(
                name,
                "Redesign or select a maintained third-party replacement.",
            )
            for name in requested_removed
        },
    }


project_imports = {"pathlib", "json", "cgi", "pipes", "telnetlib"}
audit = pep594_audit(project_imports)
pprint(audit, sort_dicts=False)

assert audit["removed_imports_found"] == ["cgi", "pipes", "telnetlib"]
assert audit["unexpectedly_still_in_stdlib"] == []

{'removed_imports_found': ['cgi', 'pipes', 'telnetlib'],
 'unexpectedly_still_in_stdlib': [],
 'guidance': {'cgi': 'Use urllib.parse for query parsing and email.message for '
                     'MIME data.',
              'pipes': 'Use shlex.quote and subprocess with argument lists.',
              'telnetlib': 'Use a maintained Telnet client package or replace '
                           'the protocol.'}}


## Problem 24 — Capstone: a typed, immutable, gracefully terminating event pipeline

Combine several Python 3.13 features:

- validate unknown objects with `TypeIs`;
- represent accepted events as frozen dataclasses;
- update immutable state with `copy.replace()`;
- encode arbitrary payloads with the Z85 envelope from Problem 17;
- terminate an `asyncio.Queue` with `shutdown()` rather than a sentinel.

Invalid objects should be rejected without crashing the pipeline.

In [31]:
from copy import replace as copy_replace
from dataclasses import dataclass
from typing import TypeIs, TypedDict


class IncomingEvent(TypedDict):
    event_id: str
    payload: bytes


def is_incoming_event(value: object) -> TypeIs[IncomingEvent]:
    return (
        isinstance(value, dict)
        and isinstance(value.get("event_id"), str)
        and isinstance(value.get("payload"), bytes)
    )


@dataclass(frozen=True, slots=True)
class ProcessedEvent:
    event_id: str
    encoded_payload: bytes
    status: str = "accepted"


async def event_pipeline(values: list[object]) -> tuple[list[ProcessedEvent], int]:
    queue_: asyncio.Queue[object] = asyncio.Queue()
    accepted: list[ProcessedEvent] = []
    rejected = 0

    async def producer() -> None:
        for value in values:
            await queue_.put(value)
        queue_.shutdown()

    async def consumer() -> None:
        nonlocal rejected
        while True:
            try:
                value = await queue_.get()
            except asyncio.QueueShutDown:
                return
            try:
                if not is_incoming_event(value):
                    rejected += 1
                    continue

                event = ProcessedEvent(
                    event_id=value["event_id"],
                    encoded_payload=z85_pack(value["payload"]),
                )
                verified = z85_unpack(event.encoded_payload)
                if verified != value["payload"]:
                    event = copy_replace(event, status="corrupt")
                accepted.append(event)
            finally:
                queue_.task_done()

    async with asyncio.TaskGroup() as group:
        group.create_task(producer())
        group.create_task(consumer())

    await queue_.join()
    return accepted, rejected


events, rejected_count = await event_pipeline(
    [
        {"event_id": "evt-1", "payload": b"hello"},
        {"event_id": 999, "payload": b"bad-id"},
        {"event_id": "evt-2", "payload": bytes(range(9))},
        "not even a mapping",
    ]
)

print("accepted:", events)
print("rejected:", rejected_count)

assert [event.event_id for event in events] == ["evt-1", "evt-2"]
assert all(event.status == "accepted" for event in events)
assert rejected_count == 2

accepted: [ProcessedEvent(event_id='evt-1', encoded_payload=b'00005xK#0@zVx+q', status='accepted'), ProcessedEvent(event_id='evt-2', encoded_payload=b'00009009c61o!#m2MK&8', status='accepted')]
rejected: 2


## Additional advanced exercises — with concise solutions

These are deliberately smaller variations for extra practice.

### Exercise A — `re.PatternError`

Compile an invalid regular expression and catch the clearer Python 3.13 exception name while preserving compatibility with the old alias.

In [32]:
import re

try:
    re.compile(r"(unclosed")
except re.PatternError as exc:
    print(type(exc).__name__ + ":", exc)
    assert re.error is re.PatternError

PatternError: missing ), unterminated subpattern at position 0


### Exercise B — Keyword `count` in `str.replace()`

Use the newly keyword-capable `count` argument to replace only the first two separators.

In [33]:
version_text = "3-13-advanced-notebook"
normalized = version_text.replace("-", ".", count=2)
print(normalized)
assert normalized == "3.13.advanced-notebook"

3.13.advanced-notebook


### Exercise C — Keyword namespaces for `eval()` and `exec()`

Make namespace intent explicit with keyword arguments.

In [34]:
globals_ns = {"__builtins__": {}}
locals_ns = {"x": 6, "y": 7}
product = eval("x * y", globals=globals_ns, locals=locals_ns)
exec("result = x + y", globals=globals_ns, locals=locals_ns)
print(product, locals_ns["result"])
assert product == 42 and locals_ns["result"] == 13

42 13


## Final checklist

- [x] Capability detection for free-threading and JIT builds
- [x] Thread-safe design patterns
- [x] `copy.replace()` and the replace protocol
- [x] `itertools.batched(strict=True)`
- [x] PEP 667 `locals()` and `frame.f_locals`
- [x] Type defaults, `TypeIs`, `ReadOnly`, and protocol introspection
- [x] Runtime/static deprecations and CLI migration
- [x] Async and threaded queue shutdown
- [x] Async completion correlation and multi-separator reads
- [x] Z85, optimized ASTs, and class metadata
- [x] Unnamed config sections, CPU-aware sizing, SQLite `dbm`
- [x] PEP 594 migration audit
- [x] Integrated capstone with tests

### Recommended next step

Run the notebook under both a regular CPython 3.13 build and, where available, a free-threaded `python3.13t` build. Compare **correctness first**, then profile representative workloads before making concurrency or JIT deployment decisions.